In [2]:
import torch
import time
import warnings
from typing import Dict, Any, List
from Solver import *
from repo_preliminary_instances import *

In [3]:



def get_all_instances():
    """Returns a list of (name, J, h, analytical_optimum)."""
    return [
        ("Field-only", *field_only(N=12)),
        ("Ferromagnet", *ferromagnet(N=12)),
        ("Antiferro Cycle", *antiferro_even_cycle(N=6)), # N=12 total spins
        ("Frustrated Triangle", *frustrated_triangle()),
        ("Max-Cut K4", *maxcut_K4()),
        ("Max-Cut C5", *maxcut_C5()),
        ("Number Partitioning", *partition_small()),
        ("SK Spin Glass", *sk_glass(N=12)),
    ]

# --- [BENCHMARKING SUITE] ---

def validate_config(cfg: SBConfig):
    """Addresses Issue 1 & 2: Ensures p_max > delta for bifurcation."""
    if cfg.p_max <= cfg.delta:
        warnings.warn(
            f"Bifurcation Warning: p_max ({cfg.p_max}) is not greater than delta ({cfg.delta}). "
            "The system may not cross the bifurcation threshold, leading to poor convergence."
        )

def run_benchmarks():
    instances = get_all_instances()
    
    print(f"{'Instance Name':<22} | {'Optimal E':>10} | {'SBM Best E':>10} | {'Gap %':>8} | {'Success%':>8} | {'Time (s)':>8}")
    print("-" * 85)

    for name, J, h, analytical_E in instances:
        # 1. Get Ground Truth (Brute Force if analytical is None)
        if analytical_E is None:
            true_E, _ = brute_force_optimum(J, h)
        else:
            true_E = analytical_E

        # 2. Configure and Run SBM
        # We set p_max = 1.5 * delta to ensure we pass the bifurcation threshold (Issue 1)
        config = SBConfig(
            n_parallel=2000,   # Large ensemble for better success rate metrics
            p_max=1.5,         # delta is 1.0 by default
            n_steps=500,
            seed=42
        )
        validate_config(config)

        start_time = time.perf_counter()
        result = solve(J, h, config)
        end_time = time.perf_counter()

        # 3. Calculate Metrics
        best_sbm_e = result.energy
        successes = torch.sum(result.all_energies <= true_E + 1e-4).item()
        success_rate = (successes / config.n_parallel) * 100
        
        
        gap = abs((best_sbm_e - true_E) / (true_E if true_E != 0 else 1.0)) * 100

        print(f"{name:<22} | {true_E:>10.3f} | {best_sbm_e:>10.3f} | {gap:>7.2f}% | {success_rate:>7.1f}% | {end_time-start_time:>8.3f}")

if __name__ == "__main__":
    # Ensure reproducibility
    torch.manual_seed(42)
    
    print("Starting Simulated Bifurcation Machine Demo Benchmarks...\n")
    run_benchmarks()

Starting Simulated Bifurcation Machine Demo Benchmarks...

Instance Name          |  Optimal E | SBM Best E |    Gap % | Success% | Time (s)
-------------------------------------------------------------------------------------
Field-only             |    -10.207 |    -10.207 |    0.00% |   100.0% |    0.140
Ferromagnet            |    -66.000 |    -66.000 |    0.00% |   100.0% |    0.109
Antiferro Cycle        |    -12.000 |    -12.000 |    0.00% |   100.0% |    0.095
Frustrated Triangle    |     -1.000 |     -1.000 |    0.00% |   100.0% |    0.089
Max-Cut K4             |     -2.000 |     -2.000 |    0.00% |   100.0% |    0.100
Max-Cut C5             |     -3.000 |     -3.000 |    0.00% |   100.0% |    0.074
Number Partitioning    |    -10.000 |    -10.000 |    0.00% |    38.3% |    0.086
SK Spin Glass          |     -6.228 |     -6.228 |    0.00% |    99.7% |    0.118
